# 02 — imzML smoke test

Goal: confirm that pyimzml can parse an imzML file end-to-end on this machine, and that we can render both a mass spectrum and an ion image from it.

This notebook doesn't use project data. It uses the official **Example_Continuous** test file from the [imzml.org example files page](https://www.ms-imaging.org/imzml/example-files-test/), distributed via the pyimzml test suite. This is a tiny 3×3 pixel dataset designed for parser testing — not biology — but if pyimzml can read it cleanly here, the imzML half of the Maillat 2026 pipeline is technically viable.

The files were downloaded to `data/raw/Example_Continuous.{imzML,ibd}` before running this notebook.

In [ ]:
# Imports for imzML smoke test
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pyimzml.ImzMLParser import ImzMLParser, getionimage

%matplotlib inline

# Path to the test imzML file (the matching .ibd must sit alongside it)
# This notebook lives in notebooks/; data lives in ../data/raw/
imzml_path = Path('..') / 'data' / 'raw' / 'Example_Continuous.imzML'
print(f"Looking for: {imzml_path.resolve()}")
print(f"File exists: {imzml_path.exists()}")
print(f"File size:   {imzml_path.stat().st_size:,} bytes")

# Open the imzML file
p = ImzMLParser(str(imzml_path))
print("\nImzMLParser instantiated successfully.")
print(f"Number of spectra in file: {len(p.coordinates)}")

In [ ]:
# Inspect what's inside this file
print("=" * 60)
print("Coordinates (x, y, z) of all spectra:")
for i, coord in enumerate(p.coordinates):
    print(f"  spectrum {i}: {coord}")

print("\n" + "=" * 60)
print("Selected file-level metadata (p.imzmldict):")
for k, v in p.imzmldict.items():
    print(f"  {k}: {v}")

# Pull spectrum 0 and look at its shape
mzs, intensities = p.getspectrum(0)
mzs = np.array(mzs)
intensities = np.array(intensities)

print("\n" + "=" * 60)
print("Spectrum 0 (first pixel):")
print(f"  m/z array shape:       {mzs.shape}")
print(f"  intensity array shape: {intensities.shape}")
print(f"  m/z range:             {mzs.min():.2f} to {mzs.max():.2f}")
print(f"  intensity range:       {intensities.min():.2f} to {intensities.max():.2f}")
print(f"  number of m/z points:  {len(mzs):,}")

In [ ]:
# Plot the full spectrum from pixel (1, 1, 1)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mzs, intensities, linewidth=0.8)
ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title(f'Spectrum from pixel {p.coordinates[0]} '
             f'({len(mzs):,} m/z points, range {mzs.min():.1f}–{mzs.max():.1f})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Identify and print the top 5 peaks
top_n = 5
top_indices = np.argsort(intensities)[-top_n:][::-1]  # indices of top N by intensity
print(f"\nTop {top_n} peaks in spectrum 0:")
print(f"{'rank':>4}  {'m/z':>10}  {'intensity':>12}")
for rank, idx in enumerate(top_indices, start=1):
    print(f"{rank:>4}  {mzs[idx]:>10.4f}  {intensities[idx]:>12.4f}")

In [ ]:
# Pick the strongest peak in spectrum 0 and render its ion image
target_mz = mzs[top_indices[0]]
tolerance = 0.1  # m/z window: integrate intensities within ±0.1 of target_mz
print(f"Building ion image for m/z = {target_mz:.4f} ± {tolerance}")

# getionimage returns a 2D numpy array (height x width) with one intensity per pixel
ion_img = getionimage(p, target_mz, tol=tolerance)
print(f"Ion image shape: {ion_img.shape}")
print(f"Ion image values:\n{ion_img}")

# Plot it. Tiny grid -> turn off interpolation so pixels stay sharp.
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(ion_img, cmap='viridis', interpolation='nearest')
ax.set_title(f'Ion image at m/z = {target_mz:.4f} ± {tolerance}\n'
             f'(3 × 3 pixel synthetic dataset)')
ax.set_xlabel('x (pixel)')
ax.set_ylabel('y (pixel)')
plt.colorbar(im, ax=ax, label='Intensity')
plt.tight_layout()

# Save to results/figures/
fig_path = Path('..') / 'results' / 'figures' / '02_imzml_ion_image.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved to {fig_path.resolve()}")

## Conclusion

The imzML pipeline runs end-to-end on this machine:

- **Tool**: pyimzml 1.5.5
- **Test file**: `Example_Continuous` from imzml.org via the pyimzml test suite (3×3 pixels, 8,399 m/z points per pixel, m/z range 100–800)
- **Successful operations**:
  - File loaded and parsed without error
  - All 9 pixel coordinates and file-level metadata read from the XML
  - Single spectrum extracted, top peaks identified (m/z 152.9 and 329.0 dominant)
  - Ion image rendered for m/z 152.9167 ± 0.1 across the 3×3 grid
  - Figure saved to `results/figures/02_imzml_ion_image.png`

This confirms the MSI data half of the Maillat 2026 pipeline is technically viable on this hardware. Next: switch to a real biology dataset (e.g. EUCLID pregnancy tutorial) instead of synthetic test data.

### Notes for later

- `pyimzml.ImzMLParser.getionimage(p, target_mz, tol)` is the workhorse for extracting per-pixel intensities at a given m/z. Used here with a fixed tolerance; real analyses use peak picking or PPM-based windows.
- The `.imzML` file is XML metadata; the `.ibd` file holds the binary spectrum data. They must share a base filename and live in the same folder.
- For real datasets we'll want to consider memory carefully — a real tissue MSI scan can have 10⁴–10⁶ pixels, each with thousands of m/z bins.